# Day 7: Generation, Walked Through

Retrieval finds candidates. Ranking orders them. Generation turns them into an
actual answer. This notebook walks through templates, prompts, formatting,
attribution, context limits, quality metrics, and the full pipeline.

Run each cell in order. Plain Python throughout, self-contained, no LLM API needed.

## Cell 1: What is generation and why it matters

Generation takes the context retrieval and ranking found and turns it into a
readable answer. Great retrieval and great ranking can still produce a bad user
experience if generation just dumps raw chunks back at the user instead of
actually answering the question. Generation is the last mile of RAG -- and the
only stage that actually talks to the user.

## Cell 2: Simple template-based generation

The simplest possible approaches: fill a template, or stitch chunks together.

In [ ]:
ranked_chunks = [
    {"metadata": {"title": "What is RAG",
                   "text": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer."},
     "final_score": 0.92},
    {"metadata": {"title": "Why RAG Matters",
                   "text": "RAG matters because language models can be outdated. Retrieval lets them use fresh information."},
     "final_score": 0.78},
]

def generate_template(ranked_chunks):
    if not ranked_chunks:
        return "I don't have any relevant information to answer that question."
    best = ranked_chunks[0]
    return f"Based on \"{best['metadata']['title']}\": {best['metadata']['text']}"

def generate_concatenation(query, ranked_chunks, max_chunks=3):
    if not ranked_chunks:
        return "I don't have any relevant information to answer that question."
    parts = [f"Regarding \"{query}\", here's what I found:"]
    for i, c in enumerate(ranked_chunks[:max_chunks], start=1):
        parts.append(f"{i}. From \"{c['metadata']['title']}\": {c['metadata']['text']}")
    return "\n\n".join(parts)

query = "What is RAG and why does it matter?"
print("--- Template-based ---")
print(generate_template(ranked_chunks))
print("\n--- Concatenation-based ---")
print(generate_concatenation(query, ranked_chunks))

## Cell 3: Prompt engineering techniques

The prompt we'd send to a real LLM shapes answer quality just as much as the
context itself. Let's build one with a system prompt, context, and the question.

In [ ]:
def build_context_block(chunks):
    return "\n\n".join(f"[{i}] Source: {c['metadata']['title']}\n{c['metadata']['text']}" for i, c in enumerate(chunks, 1))

def system_prompt():
    return ("You are a helpful assistant that answers questions using ONLY the provided context. "
            "If the context doesn't contain enough information, say so. Cite sources as [1], [2], etc.")

def question_answering_prompt(query, context_block):
    return f"{system_prompt()}\n\nContext:\n{context_block}\n\nQuestion: {query}\n\nAnswer:"

def chain_of_thought_prompt(query, context_block):
    return (f"{system_prompt()}\n\nContext:\n{context_block}\n\nQuestion: {query}\n\n"
            f"First, think step by step about which sources are relevant. Then write a final answer.\n\nReasoning:")

context_block = build_context_block(ranked_chunks)
print(question_answering_prompt(query, context_block))
print("\n" + "="*40 + "\n")
print(chain_of_thought_prompt(query, context_block))

## Cell 4: Answer formatting

The same answer content can be shown very differently depending on the audience.

In [ ]:
import json

answer = "RAG retrieves relevant documents before generating an answer, which helps it stay accurate and current."
sources = ["What is RAG", "Why RAG Matters"]
confidence = 0.85

print("Plain text:\n", f"{answer}\n\nSources: {', '.join(sources)}")
print("\nMarkdown:\n", f"## Answer\n\n{answer}\n\n### Sources\n" + "\n".join(f"- {s}" for s in sources))
print("\nStructured JSON:\n", json.dumps({"answer": answer, "sources": sources, "confidence": confidence}, indent=2))

## Cell 5: Source attribution

Track exactly which chunk backs which claim, with a quote and a confidence score.

In [ ]:
def quote_relevant_section(chunk, max_length=80):
    text = chunk["metadata"]["text"]
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    last_space = truncated.rfind(" ")
    return (truncated[:last_space] if last_space > 0 else truncated) + "..."

def generate_attributed_answer(query, ranked_chunks):
    if not ranked_chunks:
        return f"I don't have any sourced information to answer \"{query}\"."
    lines = [f"Regarding \"{query}\":\n"]
    for c in ranked_chunks:
        quote = quote_relevant_section(c)
        confidence = c.get("final_score", 0.0)
        lines.append(f'  "{quote}" -- [{c["metadata"]["title"]}] (confidence: {confidence:.0%})')
    return "\n".join(lines)

print(generate_attributed_answer(query, ranked_chunks))
print("\nNo sources case:")
print(generate_attributed_answer("Unrelated question", []))

## Cell 6: Manage context

A real LLM has a limited context window. When there's more material than space,
keep the most relevant chunks first and drop the rest.

In [ ]:
def select_top_chunks_by_budget(ranked_chunks, max_total_chars):
    selected, total = [], 0
    for c in ranked_chunks:
        size = len(c["metadata"]["text"])
        if total + size > max_total_chars:
            continue
        selected.append(c)
        total += size
    return selected

more_chunks = ranked_chunks + [
    {"metadata": {"title": "What is a Vector Database", "text": "A vector database stores embeddings and allows fast similarity search across huge collections."}, "final_score": 0.55},
]
total_available = sum(len(c["metadata"]["text"]) for c in more_chunks)
budget = 150
selected = select_top_chunks_by_budget(more_chunks, budget)
print(f"Total available: {total_available} chars, budget: {budget} chars")
print(f"Selected {len(selected)} of {len(more_chunks)} chunks:", [c["metadata"]["title"] for c in selected])

## Cell 7: Measure quality

Does the answer address the question, stay grounded in sources, and read clearly?

In [ ]:
STOPWORDS = {"what", "is", "a", "an", "the", "are", "of", "in", "to", "and", "for", "how", "do", "does", "i", "it", "why"}

def keywords(text):
    return {w.strip("?.,!").lower() for w in text.split()} - STOPWORDS

def factual_consistency(answer, source_texts):
    answer_words = keywords(answer)
    if not answer_words:
        return 1.0
    source_words = set()
    for t in source_texts:
        source_words |= keywords(t)
    return len(answer_words & source_words) / len(answer_words)

source_texts = [c["metadata"]["text"] for c in ranked_chunks]
good_answer = "RAG retrieves relevant documents before generating an answer, helping it stay current."
fabricated_answer = "RAG was invented in 2019 and won several major international awards for innovation."

print(f"Good answer consistency:       {factual_consistency(good_answer, source_texts):.0%}")
print(f"Fabricated answer consistency: {factual_consistency(fabricated_answer, source_texts):.0%}")

## Cell 8: Advanced techniques

Expressing uncertainty and producing structured comparisons.

In [ ]:
def express_uncertainty(answer, confidence_score):
    if confidence_score >= 0.7:
        return answer
    if confidence_score >= 0.4:
        return f"I'm not fully certain, but based on the available information: {answer}"
    return f"I have low confidence in this answer, and you should verify it independently: {answer}"

for conf in [0.9, 0.5, 0.2]:
    print(f"confidence={conf}: {express_uncertainty('RAG improves answer accuracy.', conf)}")

def structured_comparison(items):
    headers = list(items[0].keys())
    rows = ["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    rows += ["| " + " | ".join(str(item[h]) for h in headers) + " |" for item in items]
    return "\n".join(rows)

print("\n" + structured_comparison([
    {"Database": "Pinecone", "Hosting": "Managed"},
    {"Database": "Chroma", "Hosting": "Local"},
]))

## Cell 9: Complete pipeline example

Retrieve, rank, and generate -- all three stages in one flow, on a tiny in-notebook
corpus.

In [ ]:
import math

CONCEPTS = {
    "programming": ["python", "programming", "language", "readable"],
    "retrieval":   ["rag", "retrieval", "retrieves", "generation"],
    "freshness":   ["outdated", "fresh", "matters"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

def cosine_similarity(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    ma, mb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot/(ma*mb) if ma and mb else 0.0

documents = [
    {"title": "What is RAG", "text": "RAG stands for Retrieval-Augmented Generation and retrieves documents before generating an answer.", "recency": 0.9},
    {"title": "Why RAG Matters", "text": "RAG matters because language models can be outdated without fresh retrieval.", "recency": 0.95},
    {"title": "What is Python", "text": "Python is a readable programming language.", "recency": 0.3},
]

def retrieve(query, top_k=3):
    q_emb = text_to_embedding(query)
    scored = [(doc, cosine_similarity(q_emb, text_to_embedding(doc["text"]))) for doc in documents]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return [{"metadata": doc, "similarity": s} for doc, s in scored[:top_k] if s > 0]

def rank(results, weights={"similarity": 0.7, "recency": 0.3}):
    scored = [{**r, "final_score": r["similarity"] * weights["similarity"] + r["metadata"]["recency"] * weights["recency"]} for r in results]
    return sorted(scored, key=lambda r: r["final_score"], reverse=True)

query = "What is RAG and why does it matter?"
retrieved = retrieve(query)
ranked = rank(retrieved)
answer = generate_concatenation(query, ranked, max_chunks=2)

print("Retrieved:", [r["metadata"]["title"] for r in retrieved])
print("Ranked:   ", [r["metadata"]["title"] for r in ranked])
print(f"\n{answer}")

## Cell 10: Key takeaways

- Generation turns retrieved, ranked context into an actual readable answer -- it's
  the only stage that talks to the user directly.
- Great retrieval and ranking can still produce a bad experience if generation just
  dumps raw text back at the user instead of synthesizing it.
- Prompt design (system prompts, few-shot examples, chain-of-thought) shapes answer
  quality as much as the context itself.
- Formatting should match the audience: plain text, Markdown, or structured JSON.
- Source attribution (quotes + confidence) is what makes an answer verifiable
  instead of something to trust blindly.
- Context windows are a real, hard limit -- select and summarize deliberately when
  there's more material than space.
- Answer quality is measurable: relevance, length, factual grounding, and clarity
  can all be checked with simple, concrete metrics.
- The full pipeline (chunk -> embed -> store -> retrieve -> rank -> generate) is
  only as strong as its weakest stage -- and generation is the stage where every
  earlier mistake either gets covered for, or gets handed straight to the user.